# Introspective MCTS (I-MCTS) | Advanced Planning & Search

In [1]:
import math
import re
from dataclasses import dataclass, field
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
insights: List[str] = []  # global insight memory
pre_insight_scores: List[float] = []   # score BEFORE insight was applied
post_insight_scores: List[float] = []  # score of NEXT expansion that used the insight

@dataclass
class Node:
    state: str
    parent: Optional['Node'] = None
    children: List['Node'] = field(default_factory=list)
    visits: int = 0
    value: float = 0.0

    def ucb1(self, c=1.41):
        if self.visits == 0: return float('inf')
        return (self.value / self.visits) + c * math.sqrt(math.log(self.parent.visits) / self.visits)

def select(n):
    while n.children: n = max(n.children, key=lambda c: c.ucb1())
    return n

def expand(node, problem):
    hint = f"\nInsights from prior reflections:\n" + "\n".join(insights[-3:]) if insights else ""
    resp = model.invoke(
        f"Problem: {problem}\nPartial solution:\n{node.state}{hint}\n\n"
        "Propose 3 next steps (numbered 1-3, one line each)."
    )
    for line in resp.content.strip().splitlines():
        step = re.sub(r"^\d+[\.)\s]", "", line.strip())
        if step: node.children.append(Node(state=f"{node.state}\n- {step}", parent=node))

def simulate(node, problem):
    resp = model.invoke(
        f"Problem: {problem}\nPartial solution:\n{node.state}\n\n"
        "Score 0.0-1.0 how promising this is. Reply with ONLY a number."
    )
    try: return max(0.0, min(1.0, float(re.search(r"[\d.]+", resp.content).group())))
    except: return 0.5

def backprop(node, val):
    while node: node.visits += 1; node.value += val; node = node.parent

def introspect(node, score):
    """Compare this node with its siblings and generate an insight."""
    if not node.parent or len(node.parent.children) < 2: return
    siblings = node.parent.children
    scored = [(n, n.value / max(n.visits, 1)) for n in siblings if n.visits > 0]
    if len(scored) < 2: return
    best = max(scored, key=lambda x: x[1])
    worst = min(scored, key=lambda x: x[1])
    resp = model.invoke(
        f"Better path (score {best[1]:.2f}):\n{best[0].state}\n\n"
        f"Worse path (score {worst[1]:.2f}):\n{worst[0].state}\n\n"
        "What specific aspect of the better path led to a higher score? "
        "How can we apply this insight to the next expansion? (1-2 sentences)"
    )
    pre_insight_scores.append(score)  # track score BEFORE this insight exists
    insights.append(resp.content.strip())

In [5]:
# --- Run I-MCTS ---
problem = "Solve: what is 23 * 17 step by step?"
root = Node(state="Start: compute 23 * 17")

for i in range(12):
    leaf = select(root)
    if leaf.visits > 0 and not leaf.children:
        expand(leaf, problem)
        leaf = leaf.children[0] if leaf.children else leaf
    score = simulate(leaf, problem)
    # Track: if insights existed before this expansion, record this score as post-insight
    if insights and len(post_insight_scores) < len(pre_insight_scores):
        post_insight_scores.append(score)
    backprop(leaf, score)
    introspect(leaf, score)  # <-- the introspective step
    print(f"Iter {i+1}: score={score:.2f}, insights={len(insights)}")

Iter 1: score=0.80, insights=0
Iter 2: score=1.00, insights=0
Iter 3: score=0.70, insights=1
Iter 4: score=1.00, insights=2
Iter 5: score=1.00, insights=2
Iter 6: score=0.80, insights=2
Iter 7: score=1.00, insights=2
Iter 8: score=1.00, insights=3
Iter 9: score=0.90, insights=4
Iter 10: score=0.50, insights=5
Iter 11: score=0.70, insights=6
Iter 12: score=0.50, insights=7


In [6]:
# --- Measure introspection impact ---
paired = min(len(pre_insight_scores), len(post_insight_scores))
if paired > 0:
    improved = sum(1 for i in range(paired) if post_insight_scores[i] > pre_insight_scores[i])
    print(f"\nIntrospection impact: {len(insights)} insights generated, "
          f"{improved}/{paired} ({100*improved/paired:.0f}%) of subsequent expansions "
          f"scored higher than the pre-insight node")
    for i in range(paired):
        arrow = "^" if post_insight_scores[i] > pre_insight_scores[i] else "v"
        print(f"  Insight {i+1}: before={pre_insight_scores[i]:.2f} -> after={post_insight_scores[i]:.2f} {arrow}")

print(f"\nAccumulated insights:")
for ins in insights: print(f"  - {ins}")

# --- Top insights ranked by score improvement ---
if paired > 0:
    ranked = sorted(range(paired),
                    key=lambda i: post_insight_scores[i] - pre_insight_scores[i], reverse=True)
    print(f"\nTop insights (by score improvement):")
    for rank, idx in enumerate(ranked[:3], 1):
        delta = post_insight_scores[idx] - pre_insight_scores[idx]
        print(f"  {rank}. \"{insights[idx][:80]}...\" (applied at iter {idx+2}, "
              f"score change: {delta:+.2f})")


Introspection impact: 7 insights generated, 2/6 (33%) of subsequent expansions scored higher than the pre-insight node
  Insight 1: before=0.70 -> after=1.00 ^
  Insight 2: before=1.00 -> after=1.00 v
  Insight 3: before=1.00 -> after=0.90 v
  Insight 4: before=0.90 -> after=0.50 v
  Insight 5: before=0.50 -> after=0.70 ^
  Insight 6: before=0.70 -> after=0.50 v

Accumulated insights:
  - The better path scores higher because it provides a clear strategy to simplify the multiplication using the distributive property by breaking 23 into 20 + 3, which allows for the multiplication to be handled in more manageable parts. Applying this insight to the next expansion, we should continue breaking down numbers into simpler components that facilitate easier calculations.
  - The better path scores higher because it provides a clear strategy to simplify the multiplication using the distributive property by breaking 23 into 20 + 3, which allows for the multiplication to be handled in more manage